# Iteration 2 – Naive Bayes + logistic regression (NB-LR)

| | Iteration 2 |
|---|---|
| **Model** | NB-LR: Naive Bayes word weights + logistic regression, one small model per label |
| **Libraries** | `scikit-learn`, `scipy`, `numpy`, `pandas` |
| **How text becomes numbers** | bag of words: every word and word pair is 1 if it is in the spell, else 0. Naive Bayes gives every word a weight per label |
| **Input** | description + rank + category + cantrip + casting time |
| **Output** | traits, traditions, save, area, damage types, sustained |
| **Data** | same 1,920 Foundry spells and split as iteration 1 |
| **Result** | micro-F1 **0.786** |
| **Test: only rank + description** | micro-F1 **0.720** |
| **Status** | ✅ main model |

## What changed compared to iteration 1

| | Iteration 1 | Iteration 2 |
|---|---|---|
| Model | BiGRU network, word vectors learned from scratch | NB-LR: Naive Bayes + logistic regression |
| Text cleaning | Foundry links deleted | links turned into words (`@Damage[2d6[fire]]` → `2d6 fire`) |
| Input | description + rank | description + rank + category + cantrip + casting time |
| Output | 3,705 keywords + 100 traits | 6 Foundry fields: traits, traditions, save, area, damage types, sustained |
| Threshold | fixed 0.35 | chosen on the validation set |

The test spells are the same 192 spells as in every other iteration.

In [1]:
import html
import json
import re
from collections import Counter
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
DATA_DIR = ROOT.parents[1] / "data" / "spellsdata"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
SEED = 42
MIN_LABEL_SUPPORT = 5  # traits with fewer spells cannot be learned
C = 1.0  # strength of the logistic regression; the best of 0.1 / 0.3 / 1 / 3 on the validation set
FULL_INPUT = ["rank", "category", "is_cantrip", "cast_time"]
ORIGINAL_IDEA = ["rank"]  # the original idea: the user only types the rank and the description
TRADITIONS = ["arcane", "divine", "occult", "primal"]
CLASS_TRAITS = {
    "animist", "bard", "champion", "cleric", "druid", "exemplar", "kineticist", "magus", "monk",
    "necromancer", "oracle", "psychic", "ranger", "sorcerer", "summoner", "thaumaturge", "witch", "wizard",
}

## Stage 1: text cleaning

Foundry descriptions contain inline commands. They are turned into the text a player would see instead of being deleted (iteration 1 deleted them):

| Raw | Cleaned |
|---|---|
| `@UUID[Compendium.pf2e.conditionitems.Item.Prone]` | `prone` |
| `@Damage[2d6[persistent,fire]]` | `2d6 persistent fire` |
| `@Check[reflex\|dc:20\|basic]` | `basic reflex save` |
| `@Template[cone\|distance:30]` | `30-foot cone` |

In [2]:
INLINE_PATTERN = re.compile(r"@(UUID|Damage|Check|Template|Embed|Localize)\[")
HTML_PATTERN = re.compile(r"<[^>]+>")
RANDOM_ID_PATTERN = re.compile(r"(?=.*\d)[A-Za-z0-9]{16}")

def closing_bracket(value: str, start: int) -> int:
    depth = 0
    for index in range(start, len(value)):
        if value[index] == "[":
            depth += 1
        elif value[index] == "]":
            depth -= 1
            if depth == 0:
                return index
    return len(value) - 1

def render_inline(kind: str, body: str, label: str | None) -> str:
    if label:
        return label
    if kind == "UUID":
        name = body.rsplit(".", 1)[-1]
        return " " if RANDOM_ID_PATTERN.fullmatch(name) else name
    if kind == "Damage":
        return " ".join(re.sub(r"[\[\],]", " ", body).split())
    if kind == "Check":
        parts = body.split("|")
        is_basic = any(part.split(":")[0] == "basic" for part in parts[1:])
        return f"basic {parts[0]} save" if is_basic else f"{parts[0]} check"
    if kind == "Template":
        params = dict(part.split(":", 1) for part in body.split("|") if ":" in part)
        shape = params.get("type", body.split("|")[0])
        return f"{params['distance']}-foot {shape}" if "distance" in params else shape
    return " "

def expand_inline(value: str) -> str:
    parts: list[str] = []
    position = 0
    while (match := INLINE_PATTERN.search(value, position)):
        body_end = closing_bracket(value, match.end() - 1)
        body = value[match.end():body_end]
        end, label = body_end, None
        if body_end + 1 < len(value) and value[body_end + 1] == "{":
            label_end = value.find("}", body_end + 1)
            end = len(value) - 1 if label_end == -1 else label_end
            label = value[body_end + 2:end]
        parts.append(value[position:match.start()])
        parts.append(f" {render_inline(match.group(1), body, label)} ")
        position = end + 1
    parts.append(value[position:])
    return "".join(parts)

def clean_text(value: str) -> str:
    value = expand_inline(value or "")
    value = HTML_PATTERN.sub(" ", value)
    return " ".join(html.unescape(value).lower().split())

example_raw = ("Creatures are knocked @UUID[Compendium.pf2e.conditionitems.Item.Prone] and take "
               "@Damage[2d6[persistent,fire]] damage (@Check[reflex|dc:20|basic]) in a @Template[cone|distance:30].")
print("raw:    ", example_raw)
print("cleaned:", clean_text(example_raw))

raw:     Creatures are knocked @UUID[Compendium.pf2e.conditionitems.Item.Prone] and take @Damage[2d6[persistent,fire]] damage (@Check[reflex|dc:20|basic]) in a @Template[cone|distance:30].
cleaned: creatures are knocked prone and take 2d6 persistent fire damage ( basic reflex save ) in a 30-foot cone .


## Stage 2: dataset and inputs

The same 1,920 spells as iteration 1 (every JSON file with a description and at least one trait or tradition). Besides the description, each spell has fields that a Game Master chooses in Foundry anyway:

* **rank** (1-10);
* **category**: spell, focus, ritual or impossible (the folder of the file);
* **is_cantrip**: cantrips have rank 1 in Foundry, so the rank alone does not show it;
* **cast_time**: 1/2/3 actions, reaction, free, minutes/hours (exploration) or days (downtime).

The model is trained twice: once with **all of these inputs** (`FULL_INPUT`) and once with **only rank + description** (`ORIGINAL_IDEA`, the test at the end).

In [3]:
def spell_category(path: Path) -> str:
    top_folder = path.relative_to(DATA_DIR).parts[0]
    return {"spells": "spell", "focus": "focus", "rituals": "ritual", "impossible-spells": "impossible"}[top_folder]

def cast_time_bucket(value: Any) -> str:
    text = str(value or "").strip().lower()
    if text in {"1", "2", "3", "reaction", "free"}:
        return text
    if re.fullmatch(r"\d (to|or) \d.*", text):
        return "variable"
    if "minute" in text or "hour" in text:
        return "exploration"
    if "day" in text or "week" in text:
        return "downtime"
    return "other"

def area_bucket(value: str | None) -> str:
    if not value:
        return "none"
    return value if value in {"burst", "emanation", "cone", "line"} else "other"

def load_spells(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(data_dir.rglob("*.json")):
        if path.name == "_folders.json":
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        system = payload.get("system", {})
        traits_data = system.get("traits", {})
        raw_text = system.get("description", {}).get("value", "")
        traits = sorted({str(label).casefold() for label in traits_data.get("value", []) if label})
        traditions = sorted({str(label).casefold() for label in traits_data.get("traditions", []) if label})
        if not raw_text.strip() or not (traits or traditions):
            continue
        defense = system.get("defense") or {}
        rows.append({
            "name": payload.get("name", path.stem),
            "raw_text": raw_text,
            "text": clean_text(raw_text),
            "rank": int(system.get("level", {}).get("value", 0) or 0),
            "category": spell_category(path),
            "is_cantrip": "cantrip" in traits,
            "cast_time": cast_time_bucket(system.get("time", {}).get("value")),
            "traits": traits,
            "traditions": traditions,
            "save": (defense.get("save") or {}).get("statistic") or "none",
            "area": area_bucket((system.get("area") or {}).get("type")),
            "damage_types": sorted({entry.get("type") for entry in (system.get("damage") or {}).values() if entry.get("type")}),
            "sustained": bool((system.get("duration") or {}).get("sustained")),
        })
    return pd.DataFrame(rows)

df = load_spells(DATA_DIR)
print(f"spells={len(df)}")
print(f"categories: {dict(Counter(df['category']))}")
print(f"cantrips: {int(df['is_cantrip'].sum())}")

spells=1920
categories: {'focus': 545, 'impossible': 5, 'ritual': 93, 'spell': 1277}
cantrips: 120


## Stage 3: what the model predicts

| Field | Foundry field | Type |
|---|---|---|
| `traits` | `system.traits.value` | several labels |
| `traditions` | `system.traits.traditions` | several labels (arcane, divine, occult, primal) |
| `save` | `system.defense.save.statistic` | one choice: none / fortitude / reflex / will |
| `area` | `system.area.type` | one choice: none / burst / emanation / cone / line / other |
| `damage_types` | `system.damage[*].type` | several labels |
| `sustained` | `system.duration.sustained` | yes / no |

Not predicted: class traits (`witch`, `cleric`, ...), because the description does not say which class gets a focus spell, and traits with fewer than 5 spells. Every field is stored as a 0/1 matrix (one column per label), so all fields are handled the same way.

In [4]:
trait_support = Counter(trait for labels in df["traits"] for trait in labels)
trait_labels = sorted(trait for trait, count in trait_support.items() if count >= MIN_LABEL_SUPPORT and trait not in CLASS_TRAITS)
damage_support = Counter(label for labels in df["damage_types"] for label in labels)
damage_labels = sorted(label for label, count in damage_support.items() if count >= MIN_LABEL_SUPPORT)

GROUPS = {
    "traits": trait_labels,
    "traditions": TRADITIONS,
    "save": ["none", "fortitude", "reflex", "will"],
    "area": ["none", "burst", "emanation", "cone", "line", "other"],
    "damage_types": damage_labels,
    "sustained": ["sustained"],
}
ONE_CHOICE = {"save", "area"}  # exactly one answer per spell: the most likely label is taken

def targets(frame: pd.DataFrame, group: str) -> np.ndarray:
    # 0/1 matrix: one row per spell, one column per label of the group.
    def labels_of(value: Any) -> list[str]:
        if group == "sustained":
            return ["sustained"] if value else []
        return [value] if group in ONE_CHOICE else value
    return np.array([[label in labels_of(value) for label in GROUPS[group]] for value in frame[group]], dtype=np.int64)

print({group: len(labels) for group, labels in GROUPS.items()})
print(f"class traits (chosen by the user): {sorted(set(trait_support) & CLASS_TRAITS)}")

{'traits': 68, 'traditions': 4, 'save': 4, 'area': 6, 'damage_types': 16, 'sustained': 1}
class traits (chosen by the user): ['animist', 'bard', 'champion', 'cleric', 'druid', 'magus', 'monk', 'necromancer', 'oracle', 'psychic', 'ranger', 'sorcerer', 'summoner', 'witch', 'wizard']


## Stage 4: split and scores

* **Same split as iteration 1** (`sample(frac=1, random_state=42)`, 80/10/10): 1,536 training, 192 validation and the same **192 test spells**.
* For fields with several labels, the decision threshold is chosen on the validation set. The test set is only used for the final numbers.
* **Main score** (the one in the README): micro-F1 of traits + traditions in the 100 labels of iteration 1. Class traits and rare traits count as misses, so it can be compared with iterations 0 and 1.
* Reference: always answering `concentrate, manipulate` scores 0.473.

In [5]:
def split_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_start = int(len(shuffled) * 0.8)
    valid_start = int(len(shuffled) * 0.9)
    return shuffled.iloc[:test_start], shuffled.iloc[test_start:valid_start], shuffled.iloc[valid_start:]

train_df, valid_df, test_df = split_frame(df)
frames = {"train": train_df, "valid": valid_df, "test": test_df}
Y = {split: {group: targets(frame, group) for group in GROUPS} for split, frame in frames.items()}
ITERATION1_LABELS = sorted({label for row in df.itertuples() for label in [*row.traits, *row.traditions]})

def f1_scores(predicted: np.ndarray, actual: np.ndarray) -> dict[str, float]:
    predicted, actual = np.asarray(predicted).astype(bool), np.asarray(actual).astype(bool)
    tp = (predicted & actual).sum(axis=0)
    fp = (predicted & ~actual).sum(axis=0)
    fn = (~predicted & actual).sum(axis=0)
    per_label = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    return {"micro_f1": float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
            "macro_f1": float(per_label[actual.sum(axis=0) > 0].mean()),
            "precision": float(tp.sum() / max(tp.sum() + fp.sum(), 1)),
            "recall": float(tp.sum() / max(tp.sum() + fn.sum(), 1))}

THRESHOLD_GRID = np.round(np.concatenate([[0.02, 0.03], np.arange(0.05, 0.96, 0.05)]), 2)

def tune_threshold(probabilities: np.ndarray, actual: np.ndarray) -> float:
    return float(max(THRESHOLD_GRID, key=lambda threshold: f1_scores(probabilities >= threshold, actual)["micro_f1"]))

def decide(probabilities: dict[str, np.ndarray], thresholds: dict[str, float]) -> dict[str, np.ndarray]:
    # 0/1 per label: the most likely label for save and area, every label above the threshold otherwise.
    return {group: np.eye(p.shape[1], dtype=int)[p.argmax(axis=1)] if group in ONE_CHOICE else (p >= thresholds[group]).astype(int)
            for group, p in probabilities.items()}

def labels_per_spell(decided: dict[str, np.ndarray]) -> list[dict[str, list[str]]]:
    return [{group: [label for label, on in zip(GROUPS[group], decided[group][row]) if on] for group in GROUPS}
            for row in range(len(decided["traits"]))]

def label_matrix_100(label_sets: list[set[str]]) -> np.ndarray:
    return np.array([[label in labels for label in ITERATION1_LABELS] for labels in label_sets], dtype=np.int64)

def evaluate(probabilities: dict[str, np.ndarray], thresholds: dict[str, float], split: str = "test") -> dict[str, float]:
    decided = decide(probabilities, thresholds)
    predicted = [{*spell["traits"], *spell["traditions"]} for spell in labels_per_spell(decided)]
    actual = [{*row.traits, *row.traditions} for row in frames[split].itertuples()]
    report = {"overall": f1_scores(label_matrix_100(predicted), label_matrix_100(actual))["micro_f1"]}
    for group in GROUPS:
        if group in ONE_CHOICE:
            report[group] = float((decided[group] == Y[split][group]).all(axis=1).mean())
        else:
            report[group] = f1_scores(decided[group], Y[split][group])["micro_f1"]
    return {key: round(value, 4) for key, value in report.items()}

SCORE_ROWS = {"overall": "traits + traditions, 100 labels (micro-F1)", "traits": "traits (micro-F1)",
              "traditions": "traditions (micro-F1)", "save": "save (accuracy)", "area": "area (accuracy)",
              "damage_types": "damage types (micro-F1)", "sustained": "sustained (F1)"}

def score_table(reports: dict[str, dict[str, float]]) -> pd.DataFrame:
    return pd.DataFrame({name: [report[key] for key in SCORE_ROWS] for name, report in reports.items()},
                        index=list(SCORE_ROWS.values())).round(3)

constant = label_matrix_100([{"concentrate", "manipulate"}] * len(test_df))
actual_100 = label_matrix_100([{*row.traits, *row.traditions} for row in test_df.itertuples()])
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")
print(f"reference, always [concentrate, manipulate]: {f1_scores(constant, actual_100)['micro_f1']:.3f}")

train=1536, valid=192, test=192
reference, always [concentrate, manipulate]: 0.473


## Stage 5: Naive Bayes as a text-analysis tool

The text becomes a **bag of words**: one column for every word and word pair that occurs in at least 2 training spells, with 1 if it is in the spell and 0 if not.

For every label, Naive Bayes compares how often a word occurs in spells **with** the label and **without** it. The log of that ratio,

$$r = \log\frac{p / \lVert p \rVert_1}{q / \lVert q \rVert_1},\quad p = 1 + \sum_{\text{spells with label}} x,\quad q = 1 + \sum_{\text{spells without label}} x$$

is high for words that point to the label. The words below are learned from the data, not chosen by hand.

In [6]:
vectorizer = CountVectorizer(token_pattern=r"[a-z0-9][a-z0-9'+-]*", ngram_range=(1, 2), min_df=2, binary=True)
vectorizer.fit(train_df["text"])

def nb_log_ratio(X: sparse.csr_matrix, y: np.ndarray, alpha: float = 1.0) -> np.ndarray:
    # Naive Bayes: how much more often each feature occurs in spells WITH the label than WITHOUT it (log ratio).
    p = alpha + np.asarray(X[y == 1].sum(axis=0)).ravel()
    q = alpha + np.asarray(X[y == 0].sum(axis=0)).ravel()
    return np.log((p / p.sum()) / (q / q.sum()))

def features(frame: pd.DataFrame, encoder: OneHotEncoder) -> sparse.csr_matrix:
    # 0/1 for every word and word pair + the extra inputs (rank, category, ...) as 0/1 columns.
    extra = encoder.transform(frame[encoder.feature_names_in_].astype(str))
    return sparse.hstack([vectorizer.transform(frame["text"]), extra]).tocsr().astype(np.float64)

def fit_nb_lr(X: sparse.csr_matrix, y: np.ndarray) -> tuple[np.ndarray, LogisticRegression]:
    ratio = nb_log_ratio(X, y)
    return ratio, LogisticRegression(C=C, max_iter=3000).fit(X.multiply(ratio).tocsr(), y)

def predict_nb_lr(models: list[tuple[np.ndarray, LogisticRegression]], X: sparse.csr_matrix) -> np.ndarray:
    return np.column_stack([model.predict_proba(X.multiply(ratio).tocsr())[:, 1] for ratio, model in models])

def train_nb_lr(inputs: list[str]) -> tuple[dict[str, Any], dict[str, dict[str, np.ndarray]]]:
    encoder = OneHotEncoder(handle_unknown="ignore").fit(train_df[inputs].astype(str))
    X = {split: features(frame, encoder) for split, frame in frames.items()}
    models = {group: [fit_nb_lr(X["train"], y) for y in Y["train"][group].T] for group in GROUPS}
    probabilities = {split: {group: predict_nb_lr(models[group], X[split]) for group in GROUPS} for split in ["valid", "test"]}
    thresholds = {group: tune_threshold(probabilities["valid"][group], Y["valid"][group]) for group in GROUPS if group not in ONE_CHOICE}
    return {"encoder": encoder, "models": models, "thresholds": thresholds}, probabilities

In [7]:
X_words = vectorizer.transform(train_df["text"])
word_names = vectorizer.get_feature_names_out()
print(f"words and word pairs: {len(word_names):,}")
for trait in ["fire", "mental", "healing", "incapacitation", "illusion", "teleportation", "polymorph", "concentrate"]:
    ratio = nb_log_ratio(X_words, Y["train"]["traits"][:, trait_labels.index(trait)])
    print(f"  {trait:15s} -> {', '.join(word_names[np.argsort(ratio)[::-1][:8]])}")

words and word pairs: 22,118
  fire            -> 3d6 fire, 1d4 fire, a fiery, the flame, to burn, fire as, take fire, unblinking flame
  mental          -> target's mind, becomes frightened, creature's mind, persistent mental, 1d4 mental, 6d6 mental, actions against, mind with
  healing         -> life with, elapse while, conditions that, doesn't elapse, of conditions, conditions on, healing energy, or conditions
  incapacitation  -> use hostile, is controlled, becomes fascinated, spend all, via the, attitude becomes, release what, it's holding
  illusion        -> disbelieve, to disbelieve, disbelieve the, disbelieves, the illusory, disbelieves the, your illusion, illusion of
  teleportation   -> you teleport, extradimensional container, teleport the, same planet, plane it, miles that, transport yourself, portals
  polymorph       -> and speed, level ignore, speed reduction, statistics and, ignore your, following statistics, only attacks, unless your
  concentrate     -> summon, save

## Stage 6: NB-LR model (full input)

NB-LR (Wang & Manning, 2012) trains **one small logistic regression per label**:

1. take the 0/1 columns of a spell (words, word pairs and the extra inputs as 0/1 columns);
2. multiply every column by its Naive Bayes ratio $r$ for that label;
3. a logistic regression on these columns gives the probability of the label.

Naive Bayes gives strong word evidence from little data, and the logistic regression learns how much to trust it. The same code is used for all 6 fields. For `save` and `area` the label with the highest probability is taken.

In [8]:
full, full_probabilities = train_nb_lr(FULL_INPUT)
full_report = evaluate(full_probabilities["test"], full["thresholds"])
print(f"thresholds chosen on the validation set: {full['thresholds']}")
score_table({"iteration 2 (full input)": full_report})

thresholds chosen on the validation set: {'traits': 0.05, 'traditions': 0.3, 'damage_types': 0.03, 'sustained': 0.45}


,iteration 2 (full input)
"traits + traditions, 100 labels (micro-F1)",0.786
traits (micro-F1),0.819
traditions (micro-F1),0.781
save (accuracy),0.896
area (accuracy),0.833
damage types (micro-F1),0.713
sustained (F1),0.595


### Why NB-LR? Naive Bayes alone vs logistic regression alone vs both

The same 0/1 columns, the same `C`, thresholds chosen on validation, scored on the traits of the test set. Macro-F1 is the average over labels, so it shows how well the rare traits work.

In [9]:
X = {split: features(frame, full["encoder"]) for split, frame in frames.items()}

def one_model_per_trait(make_model) -> dict[str, np.ndarray]:
    models = [make_model().fit(X["train"], y) for y in Y["train"]["traits"].T]
    return {split: np.column_stack([model.predict_proba(X[split])[:, 1] for model in models]) for split in ["valid", "test"]}

candidates = {
    "Naive Bayes alone": one_model_per_trait(lambda: MultinomialNB(alpha=0.1)),
    "logistic regression alone": one_model_per_trait(lambda: LogisticRegression(C=C, max_iter=3000)),
    "NB-LR (both)": {split: full_probabilities[split]["traits"] for split in ["valid", "test"]},
}
rows = []
for name, probabilities in candidates.items():
    threshold = tune_threshold(probabilities["valid"], Y["valid"]["traits"])
    rows.append({"model": name, **f1_scores(probabilities["test"] >= threshold, Y["test"]["traits"])})
pd.DataFrame(rows).set_index("model").round(3)

,micro_f1,macro_f1,precision,recall
model,,,,
Naive Bayes alone,0.728,0.387,0.746,0.711
logistic regression alone,0.801,0.451,0.827,0.777
NB-LR (both),0.819,0.529,0.835,0.804


## Test: the original idea – only rank + description

The original idea of the project: **the user types only the rank and the description of a spell, and the model fills in everything else.**

To test this, the same model is trained again with only rank + description as input: no category, no cantrip, no casting time. Data, split, settings and test spells stay the same.

In [10]:
original, original_probabilities = train_nb_lr(ORIGINAL_IDEA)
original_report = evaluate(original_probabilities["test"], original["thresholds"])
score_table({"full input": full_report, "only rank + description": original_report})

,full input,only rank + description
"traits + traditions, 100 labels (micro-F1)",0.786,0.720
traits (micro-F1),0.819,0.777
traditions (micro-F1),0.781,0.661
save (accuracy),0.896,0.891
area (accuracy),0.833,0.833
damage types (micro-F1),0.713,0.724
sustained (F1),0.595,0.556


### Which labels get worse without the extra inputs?

Extra mistakes on the 192 test spells per label (traits and traditions), compared with the full input.

In [11]:
full_decided = decide(full_probabilities["test"], full["thresholds"])
original_decided = decide(original_probabilities["test"], original["thresholds"])
rows = []
for group in ["traits", "traditions"]:
    actual = Y["test"][group]
    for column, label in enumerate(GROUPS[group]):
        mistakes_full = int((full_decided[group][:, column] != actual[:, column]).sum())
        mistakes_original = int((original_decided[group][:, column] != actual[:, column]).sum())
        rows.append({"label": label, "field": group, "spells in test": int(actual[:, column].sum()),
                     "mistakes, full input": mistakes_full, "mistakes, rank + description": mistakes_original,
                     "extra mistakes": mistakes_original - mistakes_full})
drops = pd.DataFrame(rows).set_index("label")
print("concentrate / manipulate (they mostly come from the casting time), extra mistakes:",
      drops.loc[["concentrate", "manipulate"], "extra mistakes"].tolist())
drops.sort_values("extra mistakes", ascending=False).head(8)

concentrate / manipulate (they mostly come from the casting time), extra mistakes: [2, 2]


,field,spells in test,"mistakes, full input","mistakes, rank + description",extra mistakes
label,,,,,
focus,traits,48,0,44,44
occult,traditions,72,30,57,27
arcane,traditions,88,37,58,21
primal,traditions,78,26,46,20
cantrip,traits,13,0,13,13
divine,traditions,48,43,50,7
vitality,traits,6,4,6,2
sonic,traits,6,1,3,2


### What the model makes of rank + description

`predict_spell(rank, description)` is the original idea as a function. Two homebrew spells and three spells from the test set (with the correct Foundry answer below them).

In [12]:
HOMEBREW = [
    ("Lava Geyser (homebrew, project proposal)", 4,
     "You channel magical energy to rupture the earth, summoning a burst of molten rock. The eruption deals massive heat damage "
     "to anyone in the area and requires you to perform complex hand gestures to sustain it. Each creature in the area must "
     "attempt a basic Reflex save."),
    ("Molten Burst (homebrew, also used in iteration 1)", 3,
     "A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage."),
]
REAL_SPELLS = ["Gust of Wind", "Suggestion", "Wordsmith"]  # from the test set; Wordsmith is a focus spell

def as_text(spell: dict[str, list[str]]) -> dict[str, str]:
    return {group: ", ".join(labels) or "–" for group, labels in spell.items()}

def example_table(predict_spell) -> pd.DataFrame:
    rows = [{"spell": name, "rank": rank, **as_text(predict_spell(rank, description))} for name, rank, description in HOMEBREW]
    for name in REAL_SPELLS:
        spell = test_df[test_df["name"] == name]
        rank = int(spell["rank"].iloc[0])
        rows.append({"spell": f"{name} (test set)", "rank": rank, **as_text(predict_spell(rank, spell["raw_text"].iloc[0]))})
        rows.append({"spell": "↳ correct answer", "rank": rank, **as_text(labels_per_spell({group: targets(spell, group) for group in GROUPS})[0])})
    return pd.DataFrame(rows).set_index("spell")

In [13]:
def predict_spell(rank: int, description: str) -> dict[str, list[str]]:
    frame = pd.DataFrame([{"text": clean_text(description), "rank": rank}])
    X = features(frame, original["encoder"])
    probabilities = {group: predict_nb_lr(original["models"][group], X) for group in GROUPS}
    return labels_per_spell(decide(probabilities, original["thresholds"]))[0]

example_table(predict_spell)

,rank,traits,traditions,save,area,damage_types,sustained
spell,,,,,,,
"Lava Geyser (homebrew, project proposal)",4,"concentrate, earth, focus, manipulate",arcane,reflex,burst,–,–
"Molten Burst (homebrew, also used in iteration 1)",3,"concentrate, fire, focus, manipulate","arcane, occult, primal",reflex,none,fire,–
Gust of Wind (test set),1,"air, concentrate, manipulate","arcane, primal",fortitude,burst,bludgeoning,–
↳ correct answer,1,"air, concentrate, manipulate","arcane, primal",fortitude,line,bludgeoning,–
Suggestion (test set),4,"concentrate, incapacitation, linguistic, manip...","arcane, occult",will,none,–,–
↳ correct answer,4,"concentrate, incapacitation, linguistic, manip...","arcane, occult",will,none,–,–
Wordsmith (test set),4,"concentrate, focus, manipulate",occult,none,none,–,–
↳ correct answer,4,"concentrate, focus, manipulate",–,none,none,–,–


## Saved model

In [14]:
joblib.dump({"vectorizer": vectorizer, "groups": GROUPS, "full_input": full, "only_rank_and_description": original},
            ARTIFACT_DIR / "model.joblib", compress=3)
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps({"full_input": full_report, "only_rank_and_description": original_report}, indent=2),
                                           encoding="utf-8")
print(f"saved {ARTIFACT_DIR / 'model.joblib'} and metrics.json")

saved /Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/iterations/iteration_2/artifacts/model.joblib and metrics.json


## Conclusions

**Iteration 2 solves the main task with a simple model.** On the same 192 test spells it reaches **0.786** (iteration 1: 0.388, always `concentrate, manipulate`: 0.473). Save is right in 90% of the spells and area in 83%.

**Naive Bayes + logistic regression is better than each part alone.** Naive Bayes alone is precise but misses many labels. Logistic regression alone is good for common traits. NB-LR is best, especially for rare traits (higher macro-F1). Naive Bayes also explains the model: `disbelieve` → illusion, `you teleport` → teleportation.

**Test: only rank + description → 0.720.**

* Everything that is written in the description stays almost the same: save, area, damage types and traits like fire, mental and healing.
* The loss comes almost only from **focus** and **cantrip**: they are not written in the description. Focus spells also have no traditions, so the traditions get worse too.
* Casting time barely matters: `concentrate` and `manipulate` lose only 2 mistakes each.
* So the original idea works, and **asking the user one extra question, "spell, cantrip or focus spell?"**, would bring back most of the difference.

**Limitations:** the test set is small (192 spells, ±0.01 is noise). Class traits must come from the user. Damage labels in Foundry are incomplete: in 43% of the cases where the text says "*type* damage", the Foundry damage field does not contain that type.